# Импорты

In [18]:
import plotly.express as px
import pandas as pd
import numpy as np

# Входные данные

Уравнение: <br>
$\displaystyle
u''_{tt}(x, t) = \frac{16}{\pi} u''_{xx}(x, t) \\
0 < x < 1 \\
0 < t < T
$

Начальные и краевые условия: <br>
$\displaystyle
u(x, 0) = 0 \\
u'_t (x, 0) = 0 \\
u(0, t) = t \\
u(1, t) = -2 t^2
$

Сетка:<br>
$\displaystyle
\{x_j, t_k\}, ~~ j = \overline{0, M}, ~ k = \overline{0, N} \\
x_j = j h \\
h = \frac{4}{M} \\
t_k = k \tau \\
\tau = \frac{T}{N} \\
$

In [19]:
a = 16 / np.pi

def g(x: float, t: float) -> float:
    return 0

def phi(x: float) -> float:
    return 0

def psi(x: float) -> float:
    return 0

def gamma_0(t: float) -> float:
    return t

def gamma_1(t: float) -> float:
    return -2 * t ** 2

l = 1
T = 5

M = 6
N = 100

h = l / M
tau = T / N

x = np.linspace(0, l, M + 1)
t = np.linspace(0, T, N + 1)
u_numeric = np.zeros((M + 1, N + 1))

# Решение задачи

Аппроксимация производных: <br>
$\displaystyle
\frac{\partial^2 u}{\partial t^2} = \frac{u_j^{k - 1} - 2 u_j^k + u_j^{k + 1}}{\tau^2} \\
\frac{\partial^2 u}{\partial x^2} = \frac{u_{j - 1}^k - 2 u_j^k + u_{j + 1}^k}{h^2} \\
\frac{\partial u}{\partial t} = \frac{u_j^{k + 1} - u_j^k}{\tau}
$

## Начальные условия

$\displaystyle
u_j^0 = 0, ~~~ j = \overline{0, M} \\
\frac{u_j^1 - u_j^0}{\tau} = 0 ~~~ j = \overline{0, M}
$

$\displaystyle
u_j^1 = 0, ~~~ j = \overline{1, M - 1}
$

In [20]:
u_numeric[:, 0] = phi(x)
u_numeric[1: M, 1] = 0

## Граничные условия

$\displaystyle
u_0^k = t_k, ~~~ k = \overline{0, N} \\
u_M^k = -2 t_k^2, ~~~ k = \overline{0, N}
$

In [21]:
u_numeric[0, :] = gamma_0(t)
u_numeric[M, :] = gamma_1(t)

## Итерация

$\displaystyle
\frac{u_j^{k - 1} - 2 u_j^k + u_j^{k + 1}}{\tau^2} = \frac{16}{\pi} \cdot \frac{u_{j - 1}^k - 2 u_j^k + u_{j + 1}^k}{h^2} \\
u_j^{k + 1} = \frac{16 \tau^2}{\pi} \cdot \frac{u_{j - 1}^k - 2 u_j^k + u_{j + 1}^k}{h^2} + 2 u_j^k - u_j^{k - 1}, ~~~ j = \overline{1, M - 1}, k = \overline{1, N - 1}
$

In [22]:
for k in range(1, N):
    for j in range(1, M):
        u_numeric[j, k + 1] = a * tau ** 2 * (u_numeric[j - 1, k] - 2 * u_numeric[j, k] + u_numeric[j + 1, k]) / (h ** 2) + 2 * u_numeric[j, k] - u_numeric[j, k - 1]

# График

In [23]:
def prep_df(u: np.ndarray, name: str, x: np.ndarray, t: np.ndarray) -> pd.DataFrame:
    x_j, t_k = np.meshgrid(x, t, indexing='ij')
    return pd.DataFrame({
        'x': x_j.flatten(),
        't': t_k.flatten(),
        'u': u.flatten(),
        'Схема': name
    })

df = pd.concat([
    prep_df(u_numeric, "Явная", x, t),
])

fig = px.line(
    df, x="x", y="u", color="Схема", 
    animation_frame="t",
    title="Сравнение профилей решения в моменты времени t"
)

fig.update_layout(xaxis_title="Координата X", yaxis_title="Значение U")
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['redraw'] = True
fig.update_yaxes(autorange=True)

fig.show()